In [2]:
import os
import time
import requests
import sqlite3
import pandas as pd
import re
from dotenv import load_dotenv
import json

import re
from thefuzz import fuzz


In [3]:
load_dotenv()
STEAM_KEY = os.getenv('STEAM_API_KEY')
DB_PATH = 'gaming_warehouse.db'

In [ ]:

if not STEAM_KEY:
    print("ERROR: No hay api")
else:
    print("Inicio descarga Steam")
    steam_apps = []
    last_appid = 0
    #ciclo apra descargar la lista completa de juegos de steam, con paginacion por appid y un delay entre peticiones para evitar bloqueos
    while True:
        url = f"https://api.steampowered.com/IStoreService/GetAppList/v1/?key={STEAM_KEY}&max_results=50000&last_appid={last_appid}"
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json().get('response', {})
            apps = data.get('apps', [])
            if not apps: break
            steam_apps.extend(apps)
            print(f"Descargando... {len(steam_apps)} registros.")
            if data.get('have_more_results'):
                last_appid = data.get('last_appid')
                time.sleep(2)
            else: break
        else:
            print(f"Error HTTP {response.status_code}")
            break
    
    #como los nombres pude variar se hace una funciona para ahcer una limpieza de ellos que coincidan 
    def normalizar_nombre(nombre):
        if not isinstance(nombre, str): return ""
        nombre = nombre.lower().strip()
        nombre = nombre.replace('™', '').replace('®', '').replace('©', '')
        nombre = re.sub(r'\s+', ' ', nombre)
        return nombre

    print("Procesando informacion en memoria...")
    diccionario_steam = {}
    #ciclo deonde se normalizan los nombres de los juegos de steam y se guardan en un diccionario con el nombre limpio como clave y una lista de appid
    for app in steam_apps:
        n_limpio = normalizar_nombre(app['name'])
        if n_limpio:
            if n_limpio not in diccionario_steam:
                diccionario_steam[n_limpio] = []
            diccionario_steam[n_limpio].append(app['appid'])

    print("Actualizando base de datos...")

    #Ciclo para actualizar la base de datos
    with sqlite3.connect(DB_PATH) as conn:
        query_select = "SELECT juego_id, titulo, categoria FROM CAT_Juego WHERE id_steam IS NULL"
        df_db = pd.read_sql_query(query_select, conn)
        
        actualizaciones = []
        for _, fila in df_db.iterrows():
            t_limpio = normalizar_nombre(fila['titulo'])
            
            #si el nombre limpio del juego de la base de datos coincide con algun nombre limpio de steam se actualiza el id_steam con el appid correspondiente, si hay varios se toma el primero
            if t_limpio in diccionario_steam:
                ids_disponibles = diccionario_steam[t_limpio]
                
                if fila['categoria'] == 0:
                    steam_id = min(ids_disponibles)
                else:
                    steam_id = min(ids_disponibles)
                
                actualizaciones.append((steam_id, fila['juego_id']))
        #si hay actualizaciones se ejecuta un update por cada juego encontrado, con ignore para evitar errores por juegos que ya tengan id_steam
        if actualizaciones:
            print(f"Sincronizando {len(actualizaciones)} IDs encontrados...")
            cursor = conn.cursor()
            cursor.executemany("UPDATE OR IGNORE CAT_Juego SET id_steam = ? WHERE juego_id = ?", actualizaciones)
            conn.commit()
            enlazados = cursor.rowcount
        else:
            enlazados = 0

    print(f"Fin. Registros actualizados: {enlazados}.")

Inicio descarga Steam
Descargando... 50000 registros.
Descargando... 100000 registros.
Descargando... 150000 registros.
Descargando... 161333 registros.
Procesando informacion en memoria...
Actualizando base de datos...
Sincronizando 5417 IDs encontrados...
Fin. Registros actualizados: 5353.


In [22]:
diccionario_steam

{'counter-strike': [10],
 'team fortress classic': [20],
 'day of defeat': [30],
 'deathmatch classic': [40],
 'half-life: opposing force': [50],
 'ricochet': [60, 1824450, 4307680],
 'half-life': [70],
 'counter-strike: condition zero': [80],
 'half-life: blue shift': [130],
 'half-life 2': [220],
 'counter-strike: source': [240],
 'day of defeat: source': [300],
 'half-life 2: deathmatch': [320],
 'half-life deathmatch: source': [360],
 'portal': [400],
 'team fortress 2': [440],
 'left 4 dead': [500],
 'left 4 dead 2': [550],
 'dota 2': [570],
 'portal 2': [620],
 'alien swarm': [630],
 'counter-strike 2': [730],
 'rag doll kung fu': [1002],
 'red orchestra: ostfront 41-45': [1200],
 'killing floor': [1250],
 'sin episodes: emergence': [1300],
 'sin gold': [1313],
 'darwinia': [1500],
 'uplink': [1510],
 'defcon': [1520],
 'multiwinia': [1530],
 'dangerous waters': [1600],
 'space empires iv deluxe': [1610],
 'jagged alliance 2 gold': [1620],
 'disciples ii: rise of the elves': [163

In [21]:
steam_apps

[{'appid': 10,
  'name': 'Counter-Strike',
  'last_modified': 1745368572,
  'price_change_number': 34672985},
 {'appid': 20,
  'name': 'Team Fortress Classic',
  'last_modified': 1745368565,
  'price_change_number': 34672985},
 {'appid': 30,
  'name': 'Day of Defeat',
  'last_modified': 1745368580,
  'price_change_number': 34672985},
 {'appid': 40,
  'name': 'Deathmatch Classic',
  'last_modified': 1745368570,
  'price_change_number': 34672985},
 {'appid': 50,
  'name': 'Half-Life: Opposing Force',
  'last_modified': 1745368539,
  'price_change_number': 34672985},
 {'appid': 60,
  'name': 'Ricochet',
  'last_modified': 1745368568,
  'price_change_number': 34672985},
 {'appid': 70,
  'name': 'Half-Life',
  'last_modified': 1745368462,
  'price_change_number': 34672985},
 {'appid': 80,
  'name': 'Counter-Strike: Condition Zero',
  'last_modified': 1745368574,
  'price_change_number': 34672985},
 {'appid': 130,
  'name': 'Half-Life: Blue Shift',
  'last_modified': 1745368541,
  'price_cha

# Limpieza de los que faltaron

In [ ]:
#Muchso juegos usan numeros romanos en su titulo lo que me provco que no encontrara todos los juegos
def estandarizar_romanos(texto):
    romanos = {
        ' i ': ' 1 ', ' ii ': ' 2 ', ' iii ': ' 3 ', ' iv ': ' 4 ', 
        ' v ': ' 5 ', ' vi ': ' 6 ', ' vii ': ' 7 ', ' viii ': ' 8 ', 
        ' ix ': ' 9 ', ' x ': ' 10 '
    }
    texto_espaciado = f" {texto} "
    for rom, arab in romanos.items():
        texto_espaciado = texto_espaciado.replace(rom, arab)
    return texto_espaciado.strip()

In [ ]:
#Quitar simbolos de los nombres para mejorar la coincidencia y suamos la de estadrizar nuemros
def limpieza_letras(nombre):
    if not isinstance(nombre, str): return ""
    nombre = nombre.lower()
    
    # Quitar marcas comerciales y estandarizar puntuacion
    nombre = re.sub(r'[™®©]', '', nombre)
    nombre = re.sub(r'[^\w\s]', ' ', nombre)
    nombre = re.sub(r'\s+', ' ', nombre).strip()
    nombre = estandarizar_romanos(nombre)
    return nombre

In [ ]:
#suar una funcion de similitud para comapara nombres ysa ber si son igaules los juegos
similitud = fuzz.token_set_ratio("Batan: Arkham City - Game of the Year Edition","Batman: Arkham City")
print(f"Similitud: {similitud}%")

Similitud: 76%


In [8]:
lista_steam_limpia = []

for app in steam_apps:
    nom_limpio = limpieza_letras(app.get('name', ''))
    if nom_limpio:
        lista_steam_limpia.append({
            'appid': app['appid'], 
            'nombre_limpio': nom_limpio, 
            'nombre_original': app['name']
        })

In [9]:
lista_steam_limpia

[{'appid': 10,
  'nombre_limpio': 'counter strike',
  'nombre_original': 'Counter-Strike'},
 {'appid': 20,
  'nombre_limpio': 'team fortress classic',
  'nombre_original': 'Team Fortress Classic'},
 {'appid': 30,
  'nombre_limpio': 'day of defeat',
  'nombre_original': 'Day of Defeat'},
 {'appid': 40,
  'nombre_limpio': 'deathmatch classic',
  'nombre_original': 'Deathmatch Classic'},
 {'appid': 50,
  'nombre_limpio': 'half life opposing force',
  'nombre_original': 'Half-Life: Opposing Force'},
 {'appid': 60, 'nombre_limpio': 'ricochet', 'nombre_original': 'Ricochet'},
 {'appid': 70, 'nombre_limpio': 'half life', 'nombre_original': 'Half-Life'},
 {'appid': 80,
  'nombre_limpio': 'counter strike condition zero',
  'nombre_original': 'Counter-Strike: Condition Zero'},
 {'appid': 130,
  'nombre_limpio': 'half life blue shift',
  'nombre_original': 'Half-Life: Blue Shift'},
 {'appid': 220,
  'nombre_limpio': 'half life 2',
  'nombre_original': 'Half-Life 2'},
 {'appid': 240,
  'nombre_lim

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    #ontener los juegos sin id_steam para evaluarlos
    query_huerfanos = "SELECT juego_id, titulo FROM CAT_Juego WHERE id_steam IS NULL"
    df_huerfanos = pd.read_sql_query(query_huerfanos, conn)
    
    total_huerfanos = len(df_huerfanos)
    print(f"Evaluar {total_huerfanos} ")
    
    actualizaciones = []
    rescatados = 0
    
    #prier ciclo para recorrer los jeugso sin id_steam
    for contador, (index_original, fila) in enumerate(df_huerfanos.iterrows(), start=1):
        if contador % 500 == 0:
            print(f"Progreso: {contador} / {total_huerfanos}")

        titulo_igdb = limpieza_letras(fila['titulo'])
        if len(titulo_igdb) < 3: 
            continue 
            
        mejor_appid = None
        mejor_score = 0
        
        # ciclo para comparar el nombre limpio del juego de la base de datos con los nombres limpios de steam usando similitud y actualizar si hay una coincidencia alta
        for steam_app in lista_steam_limpia:
            score_set = fuzz.token_set_ratio(titulo_igdb, steam_app['nombre_limpio'])
            
            #si la similitud es mayor que el mejor score encontrado se actualiza el mejor score y el appid correspondiente
            if score_set > mejor_score:
                score_final = score_set
                
                #scamaos este score para detectar casos donde el token set da un resultado alto pero el orden de las palabras es muy diferente lo que puede indicar que no son el mismo juego
                score_sort = fuzz.token_sort_ratio(titulo_igdb, steam_app['nombre_limpio'])
                # si el token set es alto pero el token sort es bajo se penaliza el score final para evitar falsos positivos como paso con AWE y Control:AWE
                if score_set >= 90 and score_sort < 65:
                    score_final -= 25 
                    
                mejor_score = score_final
                mejor_appid = steam_app['appid']
                
            if mejor_score == 100:
                break
                
        if mejor_score >= 90:
            actualizaciones.append((mejor_appid, fila['juego_id']))
            rescatados += 1
    #se hace la actualizacion
    if actualizaciones:
        cursor = conn.cursor()
        query_update = "UPDATE OR IGNORE CAT_Juego SET id_steam = ? WHERE juego_id = ?"
        cursor.executemany(query_update, actualizaciones)
        conn.commit()
        enlazados_reales = cursor.rowcount
    else:
        enlazados_reales = 0
        
print(f"Senlazados{enlazados_reales}")

Evaluar 4807 
Progreso: 500 / 4807
Progreso: 1000 / 4807
Progreso: 1500 / 4807
Progreso: 2000 / 4807
Progreso: 2500 / 4807
Progreso: 3000 / 4807
Progreso: 3500 / 4807
Progreso: 4000 / 4807
Progreso: 4500 / 4807
Senlazados1191


In [ ]:
#ya teneiendo los id solo queda descargar losd etalles de cada jeugo que podemos obtener de  la api
def descargar_detalles_steam(limite_juegos=100):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    columnas_steam = [
        ("steam_price_initial", "REAL"), ("steam_price_final", "REAL"),
        ("steam_discount_percent", "INTEGER"), ("metacritic_score", "INTEGER"),
        ("recommendations_count", "INTEGER"), ("achievements_count", "INTEGER"),
        ("steam_languages", "TEXT"), ("pc_requirements_json", "TEXT")
    ]
    
    #agregar las columnas necesarias para guardar los detalles de steam
    for col, tipo in columnas_steam:
        try:
            cursor.execute(f"ALTER TABLE CAT_Juego ADD COLUMN {col} {tipo};")
        except sqlite3.OperationalError:
            pass
    #mu qiery apra obtener los juegos que tienen id_steam pero no tienen el precio final descargado, con un limite para procesar en lotes y evitar bloqueos por la api
    cursor.execute("""
        SELECT juego_id, id_steam, titulo 
        FROM CAT_Juego 
        WHERE id_steam IS NOT NULL AND steam_price_final IS NULL 
        LIMIT ?
    """, (limite_juegos,))
    juegos = cursor.fetchall()
    
    if not juegos:
        print("No hay juegos pendientes.")
        conn.close()
        return

    print(f"Procesando {len(juegos)}")
    
    for juego_id, appid, titulo in juegos:
        url = f"https://store.steampowered.com/api/appdetails?appids={appid}&l=spanish"
        
        try:
            response = requests.get(url)
            # si todo esta ok
            if response.status_code == 200:
                data = response.json()
                
                if data and str(appid) in data and data[str(appid)]['success']:
                    info = data[str(appid)].get('data', {})
                    
                    # obtenr info y procesarla apra guardarse
                    price_info = info.get('price_overview', {})
                    p_initial = price_info.get('initial', 0) / 100.0
                    p_final = price_info.get('final', 0) / 100.0
                    discount = price_info.get('discount_percent', 0)
                    
                    metacritic = info.get('metacritic', {}).get('score')
                    recoms = info.get('recommendations', {}).get('total', 0)
                    achievs = info.get('achievements', {}).get('total', 0)
                    langs = info.get('supported_languages')
                    pc_reqs = json.dumps(info.get('pc_requirements', {}))
                    
                    cursor.execute("""
                        UPDATE CAT_Juego SET 
                            steam_price_initial = ?, steam_price_final = ?, steam_discount_percent = ?,
                            metacritic_score = ?, recommendations_count = ?, achievements_count = ?,
                            steam_languages = ?, pc_requirements_json = ?
                        WHERE juego_id = ?
                    """, (p_initial, p_final, discount, metacritic, recoms, achievs, langs, pc_reqs, juego_id))
                    print(f"{titulo}")
                else:
                    cursor.execute("UPDATE CAT_Juego SET steam_price_final = -1 WHERE juego_id = ?", (juego_id,))
                    print(f"Sin datos{titulo}")
                
                conn.commit()
                
            elif response.status_code == 429:
                print("limite")
                time.sleep(60)
                continue 
                
        except Exception as e:
            print(f"Error en {titulo}: {e}")
            
        time.sleep(1)

    conn.close()
    print("fin")

In [ ]:
#peuba 1
descargar_detalles_steam(limite_juegos=10)

Procesando 10
The Witcher 3: Wild Hunt
Portal 2
The Elder Scrolls V: Skyrim
Grand Theft Auto: San Andreas
Portal
Red Dead Redemption 2
God of War
Half-Life 2
BioShock
Assassin's Creed II
fin


In [ ]:
#Ciclo apra que desacxrgue todo
while True:
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("""
        SELECT COUNT(*) 
        FROM CAT_Juego 
        WHERE id_steam IS NOT NULL AND steam_price_final IS NULL
    """)
    pendientes = cursor.fetchone()[0]
    conn.close()
    
    if pendientes == 0:
        print("fin")
        break
        
    print(f"\n--- Progreso: Faltan {pendientes} juegos por enriquecer ---")
    
    descargar_detalles_steam(limite_juegos=200)
    
    print("fin 2")
    time.sleep(60)


--- Progreso: Faltan 6534 juegos por enriquecer ---
Procesando 200
Grand Theft Auto: Vice City
Half-Life
Mass Effect 2
BioShock Infinite
Mass Effect
Red Dead Redemption
Fallout: New Vegas
Horizon Zero Dawn
Far Cry 3
Hollow Knight
Elden Ring
Marvel's Spider-Man
Doom
Dark Souls III
Dishonored
Undertale
Borderlands 2
Life is Strange
Mass Effect 3
Call of Duty 4: Modern Warfare
Assassin's Creed IV Black Flag
Fallout 3
Sin datosMax Payne
Stardew Valley
Assassin's Creed Brotherhood
The Walking Dead
Fallout 4
Hades
Inside
Limbo
The Elder Scrolls IV: Oblivion
Cyberpunk 2077
Mafia
NieR: Automata
Ori and the Blind Forest
Batman: Arkham Knight
Star Wars: Knights of the Old Republic
Celeste
Baldur's Gate III
Middle-earth: Shadow of Mordor
The Witcher 2: Assassins of Kings
Resident Evil 2
Rise of the Tomb Raider
Call of Duty: Black Ops
Sekiro: Shadows Die Twice
Wolfenstein: The New Order
Hotline Miami
Assassin's Creed Revelations
Detroit: Become Human
Dead Space
South Park: The Stick of Truth
The 

In [ ]:

conn = sqlite3.connect(DB_PATH)

query = """
SELECT 
    COUNT(*) as Total_Juegos_En_BD,
    SUM(CASE WHEN id_steam IS NOT NULL THEN 1 ELSE 0 END) as Total_Con_ID_Steam,
    SUM(CASE WHEN steam_price_final IS NOT NULL AND steam_price_final != -1 THEN 1 ELSE 0 END) as Detalles_Descargados_Exito,
    SUM(CASE WHEN steam_price_final = -1 THEN 1 ELSE 0 END) as Juegos_Ocultos_O_Borrados,
    SUM(CASE WHEN id_steam IS NOT NULL AND steam_price_final IS NULL THEN 1 ELSE 0 END) as Pendientes_Por_Descargar
FROM CAT_Juego;
"""

df_status = pd.read_sql_query(query, conn)
conn.close()

display(df_status)

,Total_Juegos_En_BD,Total_Con_ID_Steam,Detalles_Descargados_Exito,Juegos_Ocultos_O_Borrados,Pendientes_Por_Descargar
0,10160,6544,6533,11,0
